# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', 'Unknown Dataset')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

Each entity is referenced by its `@id` to ensure consistent referencing within the dataset.

In [ ]:
# List all record sets with their @id and field @ids
print('Available record sets and fields:')
if hasattr(dataset, 'record_sets') and dataset.record_sets:
    for rs in dataset.record_sets:
        print(f"---\nRecord set: {rs.id}")
        if hasattr(rs, 'fields') and rs.fields:
            for field in rs.fields:
                print(f"   Field: {field.id} (name: {getattr(field, 'name', '')}, dtype: {getattr(field, 'data_type', '')})")
        else:
            print("   (No fields defined)")
else:
    print('No record sets found in the schema.')

### Example of browsing records for a record set by `@id`
Replace `<record_set_id>` below with the chosen record set `@id` from the overview to see sample records.

In [ ]:
# Example: Iterate over records in a record set (update `<record_set_id>` as needed)

if hasattr(dataset, 'record_sets') and dataset.record_sets:
    record_set_ids = [rs.id for rs in dataset.record_sets]
else:
    record_set_ids = []

# Preview up to 2 records from the first record set (update as needed)
if record_set_ids:
    chosen_record_set = record_set_ids[0]
    print(f"Displaying up to 2 records for record set '{chosen_record_set}':")
    record_gen = dataset.records(record_set=chosen_record_set)
    for i, record in enumerate(record_gen):
        print(record)
        if i >= 1:
            break
else:
    print('No record set IDs found to preview records.')

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. 
Entities are referenced by their `@id` fields.

In [ ]:
# Extract data from every record set
dataframes = {}
if record_set_ids:
    print(f"Loading {len(record_set_ids)} record sets:")
    for record_set_id in record_set_ids:
        print(f" - {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
        else:
            print(f"   No records found for `{record_set_id}`.")

    # Pick the first non-empty DataFrame for demonstration
    selected_record_set = None
    for k,v in dataframes.items():
        if not v.empty:
            selected_record_set = k
            break

    if selected_record_set:
        print(f"\nColumns for `{selected_record_set}`:")
        print(dataframes[selected_record_set].columns.tolist())
        display(dataframes[selected_record_set].head())
    else:
        print('No non-empty DataFrames loaded from record sets.')
else:
    print("No record_set_ids available for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping.

- **Filtering**: Select records meeting specified criteria (e.g., age > 50).
- **Normalization**: Transform a numeric field to zero mean/unit variance.
- **Grouping**: Aggregate data grouped by a categorical field for summary statistics.

All fields must be referenced by their `@id` (not just their column name).

In [ ]:
# Automatically pick a numeric field from the selected record set for demonstration
if selected_record_set is not None:
    df = dataframes[selected_record_set].copy()

    # Identify candidate numeric fields by checking for numeric dtype
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field (by @id): {numeric_field_id}")
        
        # Filtering records based on threshold
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].std() > 0 else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold} :")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() > 0 else 1)
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a categorical field (non-numeric, non-object)
        group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        if group_fields:
            group_field_id = group_fields[0]
            print(f"\nGrouping by: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean','count'])
            print(grouped_df.head())
        else:
            print('No categorical field found for grouping.')
    else:
        print('No numeric field found for EDA.')
else:
    print('No data available for EDA section.')

## 5. Visualization
Visualize distributions and relationships between fields.

We use the fields' `@id`s as column names.

In [ ]:
# Visualization section
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set is not None and numeric_fields:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If we have grouping, show a boxplot
    if group_fields:
        plt.figure(figsize=(9,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Not enough data for visualization: check previous steps.')

## 6. Conclusion
This notebook walked you through discovery, loading, and basic exploration of the FAIR² dataset defined by a Croissant schema, using the `mlcroissant` API.

**Key takeaways:**
- All dataset entities (record sets, fields, etc.) were referenced by their `@id` for reproducibility and traceability.
- Data can be efficiently loaded, filtered, normalized, grouped, and visualized using Croissant metadata and schema-aware extraction.
- For further work, dive deeper into clinical variables, perform modeling or statistical analysis, and explore more advanced visualizations as appropriate.